# Sesión 6 — Operaciones sobre columnas y transformaciones
## `.apply()`, `map()`, variables derivadas y pipelines

**Contexto clínico:** Enriquecimiento de perfiles clínicos caninos  
**Herramientas:** Python 3.10 · pandas · NumPy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Cargar el dataset desde el CSV guardado en sesiones anteriores
# Si no existe aún, lo reconstruimos
import os

ruta_csv = '../datos/perfil_clinico_canino.csv'

if os.path.exists(ruta_csv):
    df = pd.read_csv(ruta_csv)
    print(f"Dataset cargado desde CSV: {df.shape}")
else:
    # Reconstrucción mínima para que el notebook sea autónomo
    datos = {
        'paciente_id':         ['C001','C002','C003','C004','C005',
                                'C006','C007','C008','C009','C010',
                                'C011','C012','C013','C014','C015'],
        'raza':                ['Labrador','Beagle','Pastor Alemán','Bulldog','Golden',
                                'Schnauzer','Poodle','Boxer','Dálmata','Husky',
                                'Chihuahua','Rottweiler','Cocker','Doberman','Shih Tzu'],
        'edad_años':           [3,7,5,4,2,9,6,8,3,5,4,6,11,7,2],
        'peso_kg':             [28.5,12.0,35.2,22.1,30.0,8.5,6.2,29.8,24.5,27.3,
                                3.1,45.2,11.8,38.4,5.9],
        'sexo':                ['M','H','M','M','H','H','M','H','M','H',
                                'H','M','H','M','H'],
        'eritrocitos_mill_uL': [6.2,4.8,7.1,3.9,6.5,5.2,5.8,4.1,6.8,5.5,
                                6.0,7.2,3.5,6.6,5.9],
        'hemoglobina_g_dL':    [15.1,11.8,17.2,9.5,16.0,12.5,14.2,10.2,16.5,13.8,
                                14.8,17.8,8.9,16.2,14.5],
        'hematocrito_pct':     [45.2,35.4,51.8,28.6,48.0,37.5,42.6,30.8,49.5,41.4,
                                44.5,53.5,26.8,48.6,43.5],
        'VCM_fL':              [72.9,73.8,72.9,73.3,73.8,72.1,73.4,75.1,72.8,75.3,
                                74.2,74.3,76.6,73.6,73.7],
        'HCM_pg':              [24.4,24.6,24.2,24.4,24.6,24.0,24.5,24.9,24.3,25.1,
                                24.7,24.7,25.4,24.5,24.6],
        'CHCM_g_dL':           [33.4,33.3,33.2,33.2,33.3,33.3,33.3,33.1,33.3,33.3,
                                33.3,33.2,33.2,33.4,33.3],
        'leucocitos_mil_uL':   [8.5,12.3,7.2,18.6,9.1,11.4,8.8,15.2,7.9,10.3,
                                9.2,8.1,22.4,7.5,10.8],
        'neutrofilos_pct':     [68,75,65,82,70,78,67,80,66,72,69,64,85,67,71],
        'linfocitos_pct':      [22,16,25,10,20,14,24,12,25,18,21,27,8,24,20],
        'monocitos_pct':       [6,5,6,5,6,5,6,5,6,6,7,6,5,6,6],
        'eosinofilos_pct':     [3,3,3,2,3,2,2,2,2,3,2,2,1,2,2],
        'basofilos_pct':       [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
        'plaquetas_mil_uL':    [245,189,310,98,278,220,265,142,290,235,
                                255,320,88,275,248],
        'glucosa_mg_dL':       [95,102,88,145,91,110,87,132,93,98,
                                101,89,155,96,108],
        'bun_mg_dL':           [18,22,15,45,19,28,16,38,17,21,
                                20,14,52,18,23],
        'creatinina_mg_dL':    [0.9,1.1,0.8,2.8,1.0,1.3,0.9,2.1,0.8,1.0,
                                1.1,0.7,3.2,0.9,1.2],
        'alt_U_L':             [32,45,28,89,35,67,30,112,29,41,
                                38,25,125,33,48],
        'fosfatasa_alk_U_L':   [45,88,40,155,52,210,38,98,42,65,
                                72,35,188,48,91],
        'proteinas_totales_g_dL': [6.8,7.1,6.5,5.9,7.0,6.2,6.9,5.8,7.2,6.7,
                                   6.6,7.3,5.5,6.8,7.0],
        'diagnostico':         ['Normal','Anemia leve','Normal',
                                'Anemia moderada + AZO','Normal',
                                'Normal','Normal','Anemia leve + AZO',
                                'Normal','Normal',
                                'Normal','Normal',
                                'Anemia moderada + AZO + Leucocitosis',
                                'Normal','Normal']
    }
    df = pd.DataFrame(datos)
    os.makedirs('../datos', exist_ok=True)
    df.to_csv(ruta_csv, index=False)
    print(f"Dataset reconstruido y guardado: {df.shape}")

df.head(3)

## 1. Cuándo usar qué herramienta de transformación

| Herramienta | Cuándo usarla | Velocidad |
|---|---|---|
| Operación vectorizada (`+`, `*`, `np.where`) | Operaciones matemáticas simples sobre columnas | ★★★★★ |
| `.map()` | Reemplazar valores por un diccionario o función simple, en una **Serie** | ★★★★ |
| `.apply()` en Serie | Función Python sobre cada elemento de una columna | ★★★ |
| `.apply()` en DataFrame | Función que necesita **múltiples columnas** por fila | ★★ |
| Bucle `for` explícito | Casi nunca en pandas — señal de que hay una mejor forma | ★ |

Regla práctica: **empieza siempre por la opción más vectorizada posible.**  
`.apply()` es poderoso pero lento en datasets grandes — úsalo cuando no hay alternativa.

In [ ]:
# Las operaciones vectorizadas ya las conoces — son el nivel más eficiente
# Repaso rápido y casos clínicos nuevos

# Conversión de unidades — frecuente cuando integras datos de distintos laboratorios
# Hemoglobina de g/dL a mmol/L (factor de conversión: 0.6206)
df['hemoglobina_mmol_L'] = (df['hemoglobina_g_dL'] * 0.6206).round(3)

# BUN a urea total (BUN × 2.14 = urea en mg/dL)
df['urea_total_mg_dL'] = (df['bun_mg_dL'] * 2.14).round(1)

# Glucosa de mg/dL a mmol/L (factor: 0.0555)
df['glucosa_mmol_L'] = (df['glucosa_mg_dL'] * 0.0555).round(2)

# Índice de masa corporal veterinario simplificado (peso / talla²)
# Sin talla en el dataset, usamos peso como proxy de tamaño
# Ejemplo de transformación logarítmica — útil para variables muy asimétricas
df['log_peso'] = np.log(df['peso_kg']).round(3)

print("Columnas de conversión de unidades agregadas ✓")
df[['paciente_id', 'hemoglobina_g_dL', 'hemoglobina_mmol_L',
    'bun_mg_dL', 'urea_total_mg_dL',
    'glucosa_mg_dL', 'glucosa_mmol_L']].head(4)

In [ ]:
# Variables derivadas clínicamente relevantes — todas vectorizadas

# 1. HCM calculado: (Hgb / Eritrocitos) × 10 — índice eritrocitario
df['HCM_calculado_pg'] = ((df['hemoglobina_g_dL'] / df['eritrocitos_mill_uL']) * 10).round(1)

# 2. Ratio BUN/Creatinina — diferencial pre-renal vs renal
#    >20: sugestivo de azotemia pre-renal | <15: renal o post-renal
df['ratio_bun_creatinina'] = (df['bun_mg_dL'] / df['creatinina_mg_dL']).round(1)

# 3. Neutrófilos absolutos (miles/µL) — más informativo que el porcentaje
df['neutrofilos_abs_mil_uL'] = (
    df['leucocitos_mil_uL'] * (df['neutrofilos_pct'] / 100)
).round(2)

# 4. Linfocitos absolutos
df['linfocitos_abs_mil_uL'] = (
    df['leucocitos_mil_uL'] * (df['linfocitos_pct'] / 100)
).round(2)

# 5. Ratio neutrófilos/linfocitos (NLR) — marcador de inflamación sistémica
#    En medicina humana NLR >3 indica estrés/inflamación; en caninos similar
df['ratio_NLR'] = (df['neutrofilos_abs_mil_uL'] / df['linfocitos_abs_mil_uL']).round(2)

print("Variables derivadas calculadas ✓")
df[['paciente_id', 'HCM_calculado_pg', 'ratio_bun_creatinina',
    'neutrofilos_abs_mil_uL', 'linfocitos_abs_mil_uL', 'ratio_NLR']].round(2)

## 2. `.map()` — transformar una Serie con un diccionario o función

`.map()` opera sobre **una Serie** (columna) y reemplaza cada valor  
usando un diccionario de mapeo o una función.  
Es la herramienta correcta para codificaciones, etiquetas y sustituciones simples.

In [ ]:
# Caso 1: codificar variables categóricas para análisis estadístico o ML
# En ML, las variables categóricas necesitan ser numéricas

# Mapear sexo a variable binaria
mapa_sexo = {'M': 1, 'H': 0}
df['sexo_num'] = df['sexo'].map(mapa_sexo)

print("Codificación de sexo:")
print(df[['paciente_id', 'sexo', 'sexo_num']].head(6).to_string(index=False))

In [ ]:
# Caso 2: mapear razas a grupos de tamaño clínico
# Los rangos de referencia varían entre razas gigantes, grandes, medianas y pequeñas

mapa_tamaño = {
    'Chihuahua':      'Toy',
    'Poodle':         'Miniatura',
    'Shih Tzu':       'Miniatura',
    'Schnauzer':      'Miniatura',
    'Beagle':         'Mediano',
    'Cocker':         'Mediano',
    'Dálmata':        'Mediano',
    'Labrador':       'Grande',
    'Golden':         'Grande',
    'Husky':          'Grande',
    'Boxer':          'Grande',
    'Pastor Alemán':  'Grande',
    'Doberman':       'Grande',
    'Bulldog':        'Grande',
    'Rottweiler':     'Gigante'
}

df['grupo_tamaño'] = df['raza'].map(mapa_tamaño)

print("Distribución por grupo de tamaño:")
print(df['grupo_tamaño'].value_counts())
print()
print(df[['paciente_id', 'raza', 'peso_kg', 'grupo_tamaño']].to_string(index=False))

In [ ]:
# Caso 3: .map() con función lambda — transformación simple sin diccionario

# Clasificar la edad en etapas de vida canina
# Esta lógica es simple: una sola variable de entrada → una salida
df['etapa_vida'] = df['edad_años'].map(
    lambda edad: 'Cachorro'   if edad < 1   else
                 'Joven'      if edad < 3   else
                 'Adulto'     if edad < 8   else
                 'Senior'
)

print("Distribución por etapa de vida:")
print(df['etapa_vida'].value_counts())
print()
print(df[['paciente_id', 'edad_años', 'etapa_vida']].to_string(index=False))

In [ ]:
# Diferencia importante: .map() vs .replace()
# .map(): si un valor no está en el diccionario → NaN
# .replace(): si un valor no está en el diccionario → conserva el valor original

# Ejemplo con un diccionario incompleto
mapa_incompleto = {'M': 'Macho', 'H': 'Hembra'}   # completo en este caso

# Con .map() — NaN si falta la clave
print("Con .map():")
print(df['sexo'].map(mapa_incompleto).head())

# Con .replace() — conserva el original si falta la clave
print("\nCon .replace():")
print(df['sexo'].replace({'M': 'Macho'}).head())  # 'H' se conserva como 'H'

## 3. `.apply()` en Serie — cuando la lógica es demasiado compleja para vectorizar

`.apply()` llama a una función Python para **cada elemento** de la Serie.  
Úsalo cuando la lógica tiene ramas condicionales complejas o llamadas a funciones externas  
que no se pueden expresar con `np.where()` o vectorización directa.

**Advertencia de rendimiento:** en un dataset de 1 millón de filas, `.apply()` puede tardar  
10–50× más que una operación vectorizada equivalente. Siempre pregúntate primero:  
¿puedo hacer esto con `np.where()` o con operaciones aritméticas directas?

In [ ]:
# Caso donde .apply() tiene sentido: clasificación con lógica multi-nivel
# que sería muy verbosa con np.where() anidado

def clasificar_creatinina_iris(valor_creatinina):
    """
    Clasifica la creatinina según los estadios IRIS de ERC canina.
    
    IRIS (International Renal Interest Society) — estándar mundial para ERC en pequeños animales.
    
    Estadio 1: < 1.4 mg/dL (en riesgo, sin azotemia)
    Estadio 2: 1.4–2.0 mg/dL (azotemia leve)
    Estadio 3: 2.1–5.0 mg/dL (azotemia moderada)
    Estadio 4: > 5.0 mg/dL (azotemia severa)
    """
    if valor_creatinina < 1.4:
        return 'IRIS-1 (Sin azotemia)'
    elif valor_creatinina <= 2.0:
        return 'IRIS-2 (Leve)'
    elif valor_creatinina <= 5.0:
        return 'IRIS-3 (Moderada)'
    else:
        return 'IRIS-4 (Severa)'


# Aplicar la función a cada elemento de la columna
df['estadio_iris'] = df['creatinina_mg_dL'].apply(clasificar_creatinina_iris)

print("Estadificación IRIS de creatinina:")
print(df['estadio_iris'].value_counts())
print()
print(df[['paciente_id', 'creatinina_mg_dL', 'bun_mg_dL', 'estadio_iris']]\
        .sort_values('creatinina_mg_dL', ascending=False)\
        .to_string(index=False))

In [ ]:
# Nota: la función anterior SÍ se puede vectorizar con np.select()
# Esto es más eficiente que .apply() para datasets grandes
# Te lo muestro para que tengas ambas opciones

condiciones = [
    df['creatinina_mg_dL'] < 1.4,
    df['creatinina_mg_dL'].between(1.4, 2.0),
    df['creatinina_mg_dL'].between(2.1, 5.0),
    df['creatinina_mg_dL'] > 5.0
]

etiquetas = [
    'IRIS-1 (Sin azotemia)',
    'IRIS-2 (Leve)',
    'IRIS-3 (Moderada)',
    'IRIS-4 (Severa)'
]

# np.select() — el equivalente vectorizado de múltiples np.where() anidados
df['estadio_iris_vectorizado'] = np.select(condiciones, etiquetas, default='Sin clasificar')

# Verificar que son idénticos
son_iguales = (df['estadio_iris'] == df['estadio_iris_vectorizado']).all()
print(f"Resultados idénticos entre .apply() y np.select(): {son_iguales} ✓")
print("\nnp.select() es la forma más eficiente para clasificaciones multi-nivel")

## 4. `.apply()` en DataFrame con `axis=1` — funciones que usan múltiples columnas

Cuando una transformación necesita **más de una columna** para calcular un resultado,  
usas `.apply(función, axis=1)`. Cada fila completa se pasa como una Serie a la función.  

Este es el caso de uso legítimo de `.apply()` que no se puede reemplazar fácilmente  
con operaciones vectorizadas simples.

In [ ]:
def generar_alerta_clinica(fila):
    """
    Genera una alerta clínica textual basada en múltiples parámetros.
    Necesita acceder a varias columnas simultáneamente — caso legítimo para apply().
    
    Parámetro:
        fila: pd.Series — una fila completa del DataFrame
    
    Retorna:
        str — descripción de la alerta más prioritaria
    """
    alertas = []
    
    # Serie roja
    if fila['hematocrito_pct'] < 25.0:
        alertas.append('ANEMIA SEVERA')
    elif fila['hematocrito_pct'] < 37.0:
        alertas.append('Anemia')
    
    # Azotemia con clasificación IRIS
    if fila['creatinina_mg_dL'] > 5.0:
        alertas.append('AZOTEMIA SEVERA (IRIS-4)')
    elif fila['creatinina_mg_dL'] > 2.0:
        alertas.append('Azotemia moderada (IRIS-3)')
    elif fila['creatinina_mg_dL'] > 1.4:
        alertas.append('Azotemia leve (IRIS-2)')
    
    # Serie blanca
    if fila['leucocitos_mil_uL'] > 30.0:
        alertas.append('LEUCOCITOSIS MARCADA')
    elif fila['leucocitos_mil_uL'] > 17.0:
        alertas.append('Leucocitosis')
    elif fila['leucocitos_mil_uL'] < 6.0:
        alertas.append('Leucopenia')
    
    # Plaquetas
    if fila['plaquetas_mil_uL'] < 50.0:
        alertas.append('TROMBOCITOPENIA SEVERA')
    elif fila['plaquetas_mil_uL'] < 200.0:
        alertas.append('Trombocitopenia')
    
    # Hepático
    if fila['alt_U_L'] > 200.0:
        alertas.append('HEPATOPATÍA SEVERA')
    elif fila['alt_U_L'] > 80.0:
        alertas.append('Hepatopatía')
    
    if not alertas:
        return 'Sin alertas'
    
    # Retornar todas las alertas separadas por " | "
    return ' | '.join(alertas)


# axis=1: aplicar la función a cada FILA (no a cada columna)
df['alerta_clinica'] = df.apply(generar_alerta_clinica, axis=1)

print("Alertas clínicas generadas:")
print(df[['paciente_id', 'raza', 'alerta_clinica']].to_string(index=False))

In [ ]:
# Resumen de alertas — cuántos pacientes tienen cada tipo
print("Distribución de alertas:")
print(df['alerta_clinica'].value_counts())

# Pacientes con alertas (excluyendo 'Sin alertas')
con_alerta = df[df['alerta_clinica'] != 'Sin alertas']
print(f"\nPacientes con al menos una alerta: {len(con_alerta)} de {len(df)}")

## 5. Pipeline de transformación — de datos crudos a datos analíticos

Un pipeline de transformación es una función (o cadena de funciones) que toma  
el dataset crudo y produce el dataset analítico listo para visualización o modelado.  
Es reproducible, testeable y documentado — exactamente lo que se espera en producción.

In [ ]:
def pipeline_enriquecimiento_clinico(df_raw):
    """
    Pipeline completo de enriquecimiento de perfiles clínicos caninos.
    
    Toma el DataFrame con datos crudos de laboratorio y agrega:
    - Variables derivadas calculadas
    - Conversiones de unidades
    - Clasificaciones clínicas estandarizadas
    - Alertas automáticas
    
    Parámetro:
        df_raw: DataFrame con columnas crudas del laboratorio
    
    Retorna:
        DataFrame enriquecido (el original no se modifica)
    """
    
    # .copy() es CRÍTICO — nunca modifiques el DataFrame original
    # En Java esto sería equivalente a clonar el objeto antes de modificarlo
    df = df_raw.copy()
    
    # ── BLOQUE 1: Conversiones de unidades ──────────────────────────
    df['hemoglobina_mmol_L']  = (df['hemoglobina_g_dL'] * 0.6206).round(3)
    df['glucosa_mmol_L']      = (df['glucosa_mg_dL']    * 0.0555).round(2)
    df['urea_total_mg_dL']    = (df['bun_mg_dL']        * 2.14  ).round(1)
    
    # ── BLOQUE 2: Índices eritrocitarios calculados ──────────────────
    df['HCM_calculado_pg'] = (
        (df['hemoglobina_g_dL'] / df['eritrocitos_mill_uL']) * 10
    ).round(1)
    
    # ── BLOQUE 3: Variables de serie blanca ─────────────────────────
    df['neutrofilos_abs_mil_uL'] = (
        df['leucocitos_mil_uL'] * (df['neutrofilos_pct'] / 100)
    ).round(2)
    
    df['linfocitos_abs_mil_uL'] = (
        df['leucocitos_mil_uL'] * (df['linfocitos_pct'] / 100)
    ).round(2)
    
    df['ratio_NLR'] = (
        df['neutrofilos_abs_mil_uL'] / df['linfocitos_abs_mil_uL']
    ).round(2)
    
    # ── BLOQUE 4: Ratios bioquímicos ─────────────────────────────────
    df['ratio_bun_creatinina'] = (df['bun_mg_dL'] / df['creatinina_mg_dL']).round(1)
    
    # ── BLOQUE 5: Clasificaciones clínicas ──────────────────────────
    
    # Estadio IRIS (vectorizado con np.select)
    cond_iris = [
        df['creatinina_mg_dL'] < 1.4,
        df['creatinina_mg_dL'].between(1.4, 2.0),
        df['creatinina_mg_dL'].between(2.1, 5.0),
        df['creatinina_mg_dL'] > 5.0
    ]
    etiq_iris = ['IRIS-1','IRIS-2','IRIS-3','IRIS-4']
    df['estadio_iris'] = np.select(cond_iris, etiq_iris, default='Sin clasificar')
    
    # Estado del hematocrito
    cond_hto = [
        df['hematocrito_pct'] < 25.0,
        df['hematocrito_pct'].between(25.0, 36.9),
        df['hematocrito_pct'].between(37.0, 55.0),
        df['hematocrito_pct'] > 55.0
    ]
    etiq_hto = ['Anemia severa','Anemia','Normal','Policitemia']
    df['estado_hematocrito'] = np.select(cond_hto, etiq_hto, default='Sin clasificar')
    
    # Grupo de tamaño por raza
    mapa_tamaño = {
        'Chihuahua':'Toy', 'Poodle':'Miniatura', 'Shih Tzu':'Miniatura',
        'Schnauzer':'Miniatura', 'Beagle':'Mediano', 'Cocker':'Mediano',
        'Dálmata':'Mediano', 'Labrador':'Grande', 'Golden':'Grande',
        'Husky':'Grande', 'Boxer':'Grande', 'Pastor Alemán':'Grande',
        'Doberman':'Grande', 'Bulldog':'Grande', 'Rottweiler':'Gigante'
    }
    df['grupo_tamaño'] = df['raza'].map(mapa_tamaño)
    
    # Etapa de vida
    df['etapa_vida'] = df['edad_años'].map(
        lambda e: 'Cachorro' if e < 1 else
                  'Joven'    if e < 3 else
                  'Adulto'   if e < 8 else
                  'Senior'
    )
    
    # ── BLOQUE 6: Alertas clínicas (apply multi-columna) ────────────
    df['alerta_clinica'] = df.apply(generar_alerta_clinica, axis=1)
    
    # ── BLOQUE 7: Score de gravedad numérico ────────────────────────
    # Cuántas alteraciones mayores tiene el paciente (0–5)
    df['score_gravedad'] = (
        (df['hematocrito_pct']   < 37.0).astype(int) +
        (df['creatinina_mg_dL']  > 1.4 ).astype(int) +
        (df['leucocitos_mil_uL'] > 17.0).astype(int) +
        (df['plaquetas_mil_uL']  < 200.0).astype(int) +
        (df['alt_U_L']           > 80.0).astype(int)
    )
    
    return df


# Ejecutar el pipeline
df_analitico = pipeline_enriquecimiento_clinico(df)

print(f"DataFrame crudo:     {df.shape[1]} columnas")
print(f"DataFrame analítico: {df_analitico.shape[1]} columnas")
print(f"Variables agregadas: {df_analitico.shape[1] - df.shape[1]}")

In [ ]:
# Verificar el resultado del pipeline
print("Columnas nuevas en el DataFrame analítico:")
cols_nuevas = [c for c in df_analitico.columns if c not in df.columns]
for col in cols_nuevas:
    print(f"  {col}")

In [ ]:
# Vista del DataFrame analítico — reporte ejecutivo
print("REPORTE EJECUTIVO — Dataset analítico")
print("=" * 55)

# Score de gravedad
print("\nDistribución por score de gravedad:")
print(df_analitico['score_gravedad'].value_counts().sort_index())

# Estado del hematocrito
print("\nEstado del hematocrito:")
print(df_analitico['estado_hematocrito'].value_counts())

# Estadio IRIS
print("\nEstadio IRIS de función renal:")
print(df_analitico['estadio_iris'].value_counts())

# Etapa de vida
print("\nDistribución por etapa de vida:")
print(df_analitico['etapa_vida'].value_counts())

In [ ]:
# Guardar el DataFrame analítico enriquecido
df_analitico.to_csv('../datos/perfil_clinico_analitico.csv', index=False)
print("Dataset analítico guardado ✓")
print(f"Ruta: ../datos/perfil_clinico_analitico.csv")
print(f"Shape: {df_analitico.shape}")

In [ ]:
# 1. categoria_nlr con np.select()
cond_nlr = [
    df_analitico['ratio_NLR'] <= 3.5,
    df_analitico['ratio_NLR'].between(3.5, 6.0),
    df_analitico['ratio_NLR'].between(6.0, 9.0),
    df_analitico['ratio_NLR'] > 9.0
]
etiq_nlr = ['Normal', 'Inflamación leve', 'Inflamación moderada', 'Inflamación severa']

df_analitico['categoria_nlr'] = np.select(cond_nlr, etiq_nlr, default='Sin clasificar')

print("Distribución NLR:")
print(df_analitico['categoria_nlr'].value_counts())
print(df_analitico[['paciente_id', 'ratio_NLR', 'categoria_nlr']]\
        .sort_values('ratio_NLR', ascending=False).to_string(index=False))


# 2. perfil_hepatico con .apply() — necesita dos columnas
def clasificar_perfil_hepatico(fila):
    alt_alta = fila['alt_U_L']           > 80.0
    fa_alta  = fila['fosfatasa_alk_U_L'] > 200.0
    
    if alt_alta and fa_alta:
        return 'Mixto'
    elif alt_alta:
        return 'Hepatocelular'
    elif fa_alta:
        return 'Colestásico'
    else:
        return 'Normal'

df_analitico['perfil_hepatico'] = df_analitico.apply(clasificar_perfil_hepatico, axis=1)

print("\nDistribución de perfil hepático:")
print(df_analitico['perfil_hepatico'].value_counts())


# 3. riesgo_hemorragico — vectorizado con condición compuesta
trombocitopenia_severa    = df_analitico['plaquetas_mil_uL'] < 80.0
trombocitopenia_con_anemia = (
    (df_analitico['plaquetas_mil_uL'] < 150.0) &
    (df_analitico['hematocrito_pct']  < 30.0)
)

df_analitico['riesgo_hemorragico'] = trombocitopenia_severa | trombocitopenia_con_anemia

print("\nPacientes con riesgo hemorrágico:")
cols_rh = ['paciente_id', 'plaquetas_mil_uL', 'hematocrito_pct', 'riesgo_hemorragico']
print(df_analitico[cols_rh].to_string(index=False))
print(f"\nTotal en riesgo: {df_analitico['riesgo_hemorragico'].sum()} de {len(df_analitico)}")